# Unit 2 — Multi-Agent Pipeline with Guardrails & Handoffs
## AI-Based Defect Reporting System
### Agentic AI & Automation — Symbiosis International University

**Learning Objectives (CO2):**
- Design multi-agent pipelines
- Implement guardrails for input validation
- Build handoff mechanisms between agents
- Triage Agent → Analysis Agent → Reporter Agent


In [ ]:
import os, sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')
print('✅ Environment loaded')

## Agent 1: Triage Agent (Guardrail + Classification)

In [ ]:
from app.agents.triage_agent import triage_defect

# Test 1: Valid defect → should pass guardrail
print('=== Test 1: Valid Defect ===')
result = triage_defect('Payment gateway crashes with 500 error on checkout')
print(f'  ✅ is_defect: {result.is_defect}')
print(f'  Category: {result.category}')
print(f'  Urgency: {result.urgency}')
print(f'  Handoff to: {result.handoff_to}')

print()
print('=== Test 2: Off-Topic (Guardrail Block) ===')
result2 = triage_defect('Tell me about the cricket match today')
print(f'  ❌ is_defect: {result2.is_defect}')
print(f'  Rejection: {result2.rejection_message}')

## Agent 2: Analysis Agent (receives handoff from Triage)

In [ ]:
from app.agents.analysis_agent import analyze_defect

defect = 'Database connection pool exhausted causing 503 errors for all users'

# Triage first
triage = triage_defect(defect)
if triage.is_defect:
    print(f'✅ Triage passed → Handing off to {triage.handoff_to}')
    print(f'Category: {triage.category}, Urgency: {triage.urgency}\n')
    
    # Analysis Agent receives the handoff
    analysis = analyze_defect(defect, session_id='nb2-demo')
    print('🧠 Analysis Agent Response:')
    print(analysis['response'][:500])
    print(f'\nTools used: {[t["tool"] for t in analysis["tool_calls_made"]]}')

## Agent 3: Reporter Agent (receives handoff from Analysis)

In [ ]:
from app.agents.reporter_agent import generate_report
from app.ml.severity_predictor import predict_severity

# Predict severity
severity_result = predict_severity(
    component=triage.category,
    error_type='ConnectionFail',
    user_impact=5,
    frequency=4,
    reproducibility=4,
)
print(f'ML Severity: {severity_result["icon"]} {severity_result["severity"]}')
print(f'Confidence: {severity_result["confidence"]:.1f}%\n')

# Reporter Agent generates final report
report = generate_report(
    defect_description=defect,
    analysis_response=analysis['response'],
    session_id='nb2-demo',
    severity=severity_result['severity'],
    component=triage.category,
    send_to_n8n=True,
)

print('📄 Generated Report Preview:')
print(report['markdown_report'][:800])

## Full Pipeline: Triage → Analysis → Predict → Report

In [ ]:
def run_full_pipeline(defect_description: str):
    """Complete multi-agent pipeline demonstration."""
    print(f'Input: {defect_description}\n')
    print('Step 1: Triage Agent...')
    triage = triage_defect(defect_description)
    print(f'  → {"PASS" if triage.is_defect else "BLOCK"}: {triage.category} | {triage.urgency}')
    
    if not triage.is_defect:
        print(f'  Rejected: {triage.rejection_message}')
        return
    
    print('Step 2: Analysis Agent (handoff)...')
    analysis = analyze_defect(defect_description, 'pipeline-demo')
    print(f'  → Analysis complete | Tools: {[t["tool"] for t in analysis["tool_calls_made"]]}')
    
    print('Step 3: ML Severity Predictor...')
    severity = predict_severity(triage.category, 'Other', 4, 3, 3)
    print(f'  → {severity["icon"]} {severity["severity"]} ({severity["confidence"]:.0f}% confident)')
    
    print('Step 4: Reporter Agent (handoff)...')
    report = generate_report(
        defect_description, analysis['response'], 'pipeline-demo',
        severity['severity'], triage.category, send_to_n8n=True
    )
    print(f'  → Report generated: {report["defect_id"]}')
    print(f'  → n8n: {report["n8n_status"][:60]}')

print('=== MULTI-AGENT PIPELINE DEMO ===')
run_full_pipeline('Search function returns wrong results when special characters used')